In [ ]:
!pip install transformers
!pip install torch
!pip install pandas
!pip install tqdm

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("sosmed_no_label.csv")  # ganti sesuai nama file kamu
df.head()

Saving sosmed_no_label.csv to sosmed_no_label (1).csv


,tanggal_publish,content,tahun,username,sumber
0,6/27/2020,screenshot kandung data pribadi sebar potensi ...,2020,Titikcoma1,X (Twitter)
1,3/24/2020,hak menyebarluaskan data duduk dlm pasal ayat ...,2020,e81n,X (Twitter)
2,5/8/2020,tdk proses dg alas data pribadi payung hukum u...,2020,ismailfahmi,X (Twitter)
3,5/2/2020,gin kerdil nyata data privasi sepele uu lindun...,2020,ragilwew,X (Twitter)
4,5/22/2020,ruu pdp cepetcepet dijadiin uu dpr data pribad...,2020,rrraaihan,X (Twitter)


In [ ]:
classifier = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier",
    tokenizer="w11wo/indonesian-roberta-base-sentiment-classifier"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: w11wo/indonesian-roberta-base-sentiment-classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
tqdm.pandas()

df['sentiment'] = df['content'].progress_apply(
    lambda x: classifier(str(x))[0]['label']
)

100%|██████████| 3927/3927 [08:27<00:00,  7.73it/s]


In [ ]:
df['score'] = df['content'].progress_apply(
    lambda x: classifier(str(x))[0]['score']
)

100%|██████████| 3927/3927 [07:39<00:00,  8.54it/s]


In [ ]:
df.to_csv("dataset_labeled_sosmed.csv", index=False)

In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
netral,2782
negatif,1124
positif,21


In [ ]:
df['sentiment'].value_counts(normalize=True) * 100

,proportion
sentiment,
neutral,70.842883
negative,28.622358
positive,0.534759


In [ ]:
mapping = {
    'positive': 'positif',
    'negative': 'negatif',
    'neutral': 'netral'
}

df['sentiment'] = df['sentiment'].map(mapping)